##### *In this notebook, I build a data collection system to gather movie-related text data from:*
- *Reddit (discussion posts)*
- *Letterboxd (user reviews)*

In [1]:
import pandas as pd
import os
import re
import time
import praw
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

##### *Data Collection Architecture*

The core of this notebook is the `MovieDataCollector` class.

This class is designed to:

1. Connect to Reddit and Letterboxd
2. Fetch movie-related posts and reviews
3. Clean and filter the collected text
4. Remove irrelevant or noisy content
5. Save both raw and cleaned datasets


In [6]:
class MovieDataCollector:
    """
    A reusable class to collect, clean and save social media data for any movie across multiple platforms.
    Supports : Reddit
    """

    
    def __init__(
        self,
        movie_name,
        keywords,
        subreddits = None,
        start_year = 2008,
        end_year = 2026
    ):
        """
        movie_name : name of the movie e.g. "Titanic"
        keywords   : user defined list of movie specific keywords used to extract only relevant sentences
        subreddits : list of subreddits to search (optional)
        start_year : earliest year to collect data (defualt 2008)
        end_year   : latest year to collect until (default 2026)
        """

        self.movie_name = movie_name
        self.keywords = [kw.lower() for kw in keywords]
        self.start_year = start_year
        self.end_year = end_year

        self.subreddits = subreddits if subreddits else [
            "movies", "TrueFilm", "flicks", "cinema", "Film", "FilmTheory", "FilmNoir", 
            "MovieDetails", "Letterboxd"
        ]

        self.base_path = f"data/{self.movie_name}"
        self.raw_path = f"{self.base_path}/raw"
        self.cleaned_path = f"{self.base_path}/cleaned"
        os.makedirs(self.raw_path, exist_ok = True)
        os.makedirs(self.cleaned_path, exist_ok = True)

    # ================================================================================
    # SHARED UTILITIES
    # ================================================================================

    def clean_text(self, text):

        if not isinstance(text, str):
            return ""

        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'[\n\t\r]', ' ', text)
        text = re.sub(r'#\w+', '', text)
        text = re.sub(r'tl;dr:?', '',text, flags = re.IGNORECASE)
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r' +', ' ', text)
        text = text.strip().lower()

        return text


    def extract_relevant_sentences(self, text):

        if not isinstance(text, str) or len(text.strip()) == 0:
            return ""

        sentences = re.split(r'(?<=[.!?])\s+', text)

        relevant = []

        for sent in sentences:
            clauses = sent.split(',')

            relevant_clauses = [
                clause.strip() for clause in clauses
                if any(kw in clause.lower() for kw in self.keywords)
            ]

            if relevant_clauses:
                joined = " ".join(relevant_clauses).strip()
                if len(joined.split()) > 5:
                    relevant.append(joined)
                    
        return " ".join(relevant)


    def clean_dataframe(self, df, text_column, use_keyword_filter = True):

        print("\nCleaning Data. . .")

        df = df.copy()

        df["clean_text"] = df[text_column].apply(self.clean_text)

        if use_keyword_filter:
            df["relevant_text"] = df["clean_text"].apply(self.extract_relevant_sentences)
            print('Keyword filter applied - Reddit mode')

            cleaned_df = df[df["relevant_text"].str.split().str.len() > 4].reset_index(drop=True)
        else:
            print("No keyword filter - Letterboxd mode")
            cleaned_df = df[df["clean_text"].str.split().str.len() > 4].reset_index(drop=True)
        
        print(f"Raw rows : {len(df)}")
        print(f"Cleaned rows : {len(cleaned_df)}")

        return cleaned_df
        
        
    # ================================================================================
    # REDDIT - STEP 1: CONNECT   
    # ================================================================================

    
    def connect_reddit(self, client_id, client_secret, user_agent):
        self.reddit = praw.Reddit(
            client_id = client_id,
            client_secret = client_secret,
            user_agent = user_agent
            )
        print("Connected to Reddit API")


    # ================================================================================
    # REDDIT - STEP 2: FETCH
    # ================================================================================

        
    def fetch_reddit(self, limit = 1000):
        all_posts = []

        for sub in self.subreddits:
            print(f"Searching r/{sub}..")

            try:
                for post in self.reddit.subreddit(sub).search(self.movie_name, sort = "new", limit = limit):
                    post_year = datetime.fromtimestamp(post.created_utc).year

                    if post_year < self.start_year or post_year > self.end_year:
                        continue

                    if not post.selftext or post.selftext.strip() == "":
                        continue

                    if post.selftext in ["[deleted]", "[removed]"]:
                        continue

                    all_posts.append({
                        "movie"    : self.movie_name,
                        "platform" : "reddit",
                        "subreddit": sub,
                        "title"    : post.title,
                        "selftext" : post.selftext,
                        "year"     : post_year,
                        "score"    : post.score,
                        "url"      : post.url
                    })

                print(f"{len(all_posts)} posts collected so far")

                time.sleep(0.5)

            except Exception as e:
                print(f"Skipping r/{sub} - Error: {e}")
                continue

        raw_df = pd.DataFrame(all_posts)
        print(f"Total raw Reddit posts: {len(raw_df)}")

        raw_file = f"{self.raw_path}/{self.movie_name.lower()}_reddit_raw.csv"
        raw_df.to_csv(raw_file, index = False)
        print(f"Raw file saved -> {raw_file}")

        return raw_df


    # ================================================================================
    # REDDIT - STEP 3: MASTER METHOD
    # ================================================================================


    def run_reddit(self, client_id, client_secret, user_agent, limit = 1000):

        self.connect_reddit(client_id, client_secret, user_agent)

        raw_df = self.fetch_reddit(limit = limit)

        cleaned_df = self.clean_dataframe(raw_df, text_column = "selftext", use_keyword_filter = True)

        cleaned_file = f"{self.cleaned_path}/{self.movie_name.lower()}_reddit_cleaned.csv"
        cleaned_df.to_csv(cleaned_file, index = False)
        print(f"Cleaned data saved -> {cleaned_file}")

        print(f"Reddit pipeline completed for {self.movie_name}")
        return cleaned_df


    # ================================================================================
    # LETTERBOXD - STEP 1: CONNECT
    # ================================================================================


    def connect_letterboxd(self):

        self.driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))

        self.driver.get('https://letterboxd.com')

        print("Waiting 15 seconds for page to load...")
        print("If you see a security check in the browser - solve it manually now!")
        time.sleep(15)
        print("Chrome browser opened for Letterboxd")


    # =================================================================================
    # LETTERBOXD - STEP 2: FETCH
    # =================================================================================


    def fetch_letterboxd(self, movie_slug, max_pages = 100):
        all_reviews = []
        page_num = 1

        while True:
            if page_num == 1:
                url = f"http://letterboxd.com/film/{movie_slug}/reviews/by/activity/"
            else:
                url = f"http://letterboxd.com/film/{movie_slug}/reviews/by/activity/page/{page_num}/"
            
            #print(f"Scraping page {page_num}")

            try:
                self.driver.get(url)
                time.sleep(2)
    
                soup = BeautifulSoup(self.driver.page_source, "html.parser")
    
                review_blocks = soup.find_all("div", class_ = "listitem")
    
                if not review_blocks:
                    print(f"No more reviews at page {page_num} - stopping")
                    break
    
                page_reviews = 0
                for block in review_blocks:
                    # extract review text
                    review_div = block.find("div", class_="body-text -prose -reset js-review-body js-collapsible-text")
                    review_text = review_div.get_text(strip=True) if review_div else ""
        
                    # extract date
                    date_span = block.find("span", class_="date")
                    date_text = date_span.get_text(strip=True) if date_span else ""
        
                    # extract star rating
                    rating_span = block.find("span", class_=re.compile(r"rating"))
                    rating_text = rating_span.get_text(strip=True) if rating_span else ""
        
                    if not review_text or not date_text:
                        continue
        
                    year_match = re.search(r'\b(20\d{2}|19\d{2})\b', date_text)
                    year = int(year_match.group()) if year_match else None
        
                    if year and (year < self.start_year or year > self.end_year):
                        continue
        
                    all_reviews.append({
                        "movie"       : self.movie_name,
                        "platform"    : "letterboxd",
                        "review_text" : review_text,
                        "date"        : date_text,
                        "year"        : year,
                        "rating"      : rating_text
                    })
                #print(f"reviews - total so far: {len(all_reviews)}")
                page_num += 1
                
                if page_num > max_pages:
                    print(f"Reached {max_pages} page limit - stopping")
                    break
    
            except Exception as e:
                print(f"Error on page {page_num}: {e}")
                break
    
        raw_df = pd.DataFrame(all_reviews)
        print(f"Total letterboxd reviews collected : {len(raw_df)}")
    
        raw_file = f"{self.raw_path}/{self.movie_name.lower()}_letterboxd_raw.csv"
        raw_df.to_csv(raw_file, index = False)
        print(f"Raw data saved -> {raw_file}")

        return raw_df

    
    # =================================================================================
    # LETTERBOXD - STEP 3: MASTER METHOD
    # =================================================================================
            

    def run_letterboxd(self, movie_slug, max_pages = 100):
        self.connect_letterboxd()

        raw_df = self.fetch_letterboxd(movie_slug, max_pages = max_pages)

        cleaned_df = self.clean_dataframe(raw_df, text_column = "review_text", use_keyword_filter = False)

        cleaned_file = f"{self.cleaned_path}/{self.movie_name.lower()}_letterboxd_cleaned.csv"
        cleaned_df.to_csv(cleaned_file, index=False)
        print(f"Cleaned data saved → {cleaned_file}")

        self.driver.quit()
        print("Browser closed")

        print(f"\nLetterboxd pipeline complete for {self.movie_name}!")
        print(f"Final rows ready for emotion analysis: {len(cleaned_df)}")
        return cleaned_df

##### *Letterboxd reviews are already movie specific so no keyword filtering needed. Only basic cleaning applied — removing URLs, symbols, special characters and short reviews under 4 words. Full review text preserved to capture all emotions.*

##### *Movie Setup*

- **Movie Name** → used for searching Reddit and naming output files  
- **Keywords** → used to filter and extract relevant discussions  
- **Letterboxd Slug** → used to access the movie’s review pages for scraping  

In [12]:
movies = [
     {
        "movie_name" : "Titanic",
        "keywords"   : [
            "titanic", "kate winslet", "leonardo dicaprio",
            "iceberg", "jack dawson", "rose dewitt", "never let go",
            "i ll never let go", "my heart will go on", "celine dion",
            "jack and rose", "the door", "door scene", "drew her like",
            "im flying", "king of the world", "cal hockley",
            "1997 film", "james cameron film", "rms titanic"
        ],
        "movie_slug" : "titanic-1997"
    },
    {
        "movie_name" : "Shawshank Redemption",
        "keywords"   : [
            "shawshank", "andy dufresne", "sllis boyd red", "morgan freeman", "tim robbins",
            "stephen king", "hope is a good thing", "get busy living", "crawled through a river of fifth",
            "zihuatanejo", "warden norton", "brooks was here", "prison film", "frank darabont",
            "1994 film", "andy and red"
        ],
        "movie_slug" : "the-shawshank-redemption" 
    },
    {
        "movie_name" : "The Dark Knight",
        "keywords"   : [
            "dark knight", "joker", "batman", "heath ledger",
            "why so serious", "harvey dent", "two face",
            "christopher nolan", "gotham", "christian bale",
            "aaron eckhart", "maggie gyllenhaal", "chaos agent",
            "pencil scene", "hospital scene", "bank robbery",
            "i m an agent of chaos", "heath ledger joker",
            "2008 film", "batman film"
        ],
        "movie_slug" : "the-dark-knight"
    },
    {
        "movie_name" : "Parasite",
        "keywords"   : [
            "parasite", "bong joon ho", "ki woo", "ki taek",
            "park family", "basement", "bunker", "smell",
            "class divide", "wealth inequality", "oscar film",
            "korean film", "jessica jingle", "ram don",
            "flood scene", "garden party", "stone scholar",
            "2019 film", "bong joon", "poverty wealth"
        ],
        "movie_slug" : "parasite-2019"
    },
    {
        "movie_name" : "Interstellar",
        "keywords"   : [
            "interstellar", "matthew mcconaughey", "cooper",
            "murph", "anne hathaway", "christopher nolan",
            "black hole", "gargantua", "wormhole", "tesseract",
            "do not go gentle", "hans zimmer", "time dilation",
            "water planet", "mann planet", "brand",
            "they ve always been there", "love transcends",
            "bookshelf scene", "2014 film"
        ],
        "movie_slug" : "interstellar"
    },
    {
        "movie_name" : "La La Land",
        "keywords"   : [
            "la la land", "ryan gosling", "emma stone",
            "mia dolan", "sebastian wilder", "damien chazelle",
            "city of stars", "jazz", "audition scene",
            "epilogue", "alternate ending", "bittersweet",
            "what a waste of a lovely night", "griffith observatory",
            "someone in the crowd", "2016 film", "oscar best picture",
            "moonlight oscars", "mia and sebastian", "whiplash director"
        ],
        "movie_slug" : "la-la-land"
    },
    {
        "movie_name" : "Schindlers List",
        "keywords"   : [
            "schindler", "oskar schindler", "holocaust",
            "liam neeson", "ralph fiennes", "amon goth",
            "concentration camp", "whoever saves one life",
            "the girl in the red coat", "jewish workers",
            "stern", "world war 2 film", "steven spielberg",
            "black and white film", "i could have done more",
            "auschwitz", "krakow", "1993 film", "itzhak stern"
        ],
        "movie_slug" : "schindlers-list"
    },
    {
        "movie_name"  : "Fight Club",
        "keywords"    : [
            "fight club", "tyler durden", "brad pitt", "edward norton",
            "narrator", "marla singer", "helena bonham carter",
            "first rule", "project mayhem", "david fincher",
            "twist ending", "soap company", "1999 film",
            "you are not your job", "his name is robert paulson",
            "i am jack", "cult classic", "anarchist", "consumerism"
        ],
        "movie_slug"  : "fight-club"
    },
    {
        "movie_name"  : "A Beautiful Mind",
        "keywords"    : [
            "a beautiful mind", "john nash", "russell crowe",
            "jennifer connelly", "ron howard", "schizophrenia",
            "princeton", "nobel prize", "alicia nash",
            "mental illness", "hallucinations", "parcher",
            "charles herman", "2001 film", "oscar best picture",
            "nash equilibrium", "genius", "mathematics",
            "real or imaginary", "edward harris"
        ],
        "movie_slug"  : "a-beautiful-mind"
    },
    {
        "movie_name"  : "Inception",
        "keywords"    : [
            "inception", "christopher nolan", "leonardo dicaprio",
            "dom cobb", "arthur", "ariadne", "joseph gordon levitt",
            "tom hardy", "ellen page", "dream within a dream",
            "spinning top", "totem", "hans zimmer",
            "limbo", "kick", "2010 film", "dream heist",
            "mal cobb", "marion cotillard", "ambiguous ending"
        ],
        "movie_slug"  : "inception"
    }
]

In [46]:
for movie in movies:
    print(f"\n{'-'*25}")
    print(f"Starting collection for : {movie['movie_name']}")

    collector = MovieDataCollector(
        movie_name = movie['movie_name'],
        keywords = movie['keywords']
    )

    collector.run_reddit(
        client_id     = "REDDIT_CLIENT_ID",
        client_secret = "CLIENT_SECRET",
        user_agent    = "MovieEmotionAtlas"
    )

    collector.run_letterboxd(movie_slug = movie['movie_slug'])

    print(f"Done with {movie['movie_name']}!")


-------------------------
Starting collection for : Titanic
Connected to Reddit API
Searching r/movies..
225 posts collected so far
Searching r/TrueFilm..
348 posts collected so far
Searching r/flicks..
444 posts collected so far
Searching r/cinema..
493 posts collected so far
Searching r/Film..
564 posts collected so far
Searching r/FilmTheory..
569 posts collected so far
Searching r/FilmNoir..
570 posts collected so far
Searching r/MovieDetails..
581 posts collected so far
Searching r/Letterboxd..
744 posts collected so far
Total raw Reddit posts: 744
Raw file saved -> data/Titanic/raw/titanic_reddit_raw.csv

Cleaning Data. . .
Keyword filter applied - Reddit mode
Raw rows : 744
Cleaned rows : 402
Cleaned data saved -> data/Titanic/cleaned/titanic_reddit_cleaned.csv
Reddit pipeline completed for Titanic
Chrome browser opened for Letterboxd
Reached 100 page limit - stopping
Total letterboxd reviews collected : 1197
Raw data saved -> data/Titanic/raw/titanic_letterboxd_raw.csv

Cleani

KeyError: 'review_text'

## ✅ Data Collection Complete
Reddit data collected and saved for all 10 movies. Letterboxd reviews to be collected locally due to Colab browser limitations.